# Output Parsers

Turning a raw `AIMessage` into the shape you actually need: plain text with `StrOutputParser`, a Python `list` with `CommaSeparatedListOutputParser`, and a Python `datetime` with `DatetimeOutputParser`.

In [ ]:
# (Optional) confirm key package versions
import importlib.metadata as md

for pkg in ["langchain", "langchain-openai", "langchain-classic"]:
    try:
        print(f"{pkg:16s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:16s} NOT INSTALLED")

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(model="gpt-4o-mini", temperature=0, seed=365, max_tokens=100)

## 1. `StrOutputParser`

The simplest parser — pulls the plain string out of an `AIMessage`'s `.content`.

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser

message_h = HumanMessage(content="Can you give me an interesting fact I probably didn't know about?")
response = chat.invoke([message_h])
response

In [ ]:
str_output_parser = StrOutputParser()
str_output_parser.invoke(response)  # same as response.content, but composable in an LCEL chain

## 2. `CommaSeparatedListOutputParser`

`get_format_instructions()` returns text you append to your prompt so the model knows how to format its reply; the parser then turns that reply into a Python `list`.

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

list_output_parser = CommaSeparatedListOutputParser()

message_h = HumanMessage(content=f'''I've recently adopted a dog. Could you suggest some dog names?

{list_output_parser.get_format_instructions()}
''')
print(message_h.content)

I've recently adopted a dog. Could you suggest some dog names?

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`



In [ ]:
response = chat.invoke([message_h])
print(response.content)

In [ ]:
list_output_parser.invoke(response)

## 3. `DatetimeOutputParser`

Parses a formatted date string into a Python `datetime`. In LangChain v1, this parser moved out of `langchain.output_parsers` into the `langchain-classic` package (already in `requirements.txt`) — import it from `langchain_classic.output_parsers` instead.

In [ ]:
from langchain_classic.output_parsers import DatetimeOutputParser

date_output_parser = DatetimeOutputParser()

message_h = HumanMessage(content=f'''When was the Danish poet Piet Hein born?
{date_output_parser.get_format_instructions()}
''')
print(message_h.content)

When was the Danish poet Piet Hein born?
Write a datetime string that matches the following pattern: '%Y-%m-%dT%H:%M:%S.%fZ'.

Examples: 2023-07-04T14:30:00.000000Z, 1999-12-31T23:59:59.999999Z, 2025-01-01T00:00:00.000000Z

Return ONLY this string, no other words!



In [ ]:
response = chat.invoke([message_h])
print(response.content)

1905-12-16T00:00:00.000000Z


In [ ]:
date_output_parser.invoke(response)

datetime.datetime(1905, 12, 16, 0, 0)